# SD3.5 CityPersons Simple Pedestrian Insertion

Notebook toi gian de dung Stable Diffusion 3.5 Medium img2img tao anh CityPersons co them pedestrian. Output duoc luu vao `/kaggle/working/sd35_simple_augmented`.

## 1. Install Dependencies

Giu nguyen CUDA/PyTorch co san tren Kaggle, chi cai cac thu vien diffusion can thiet.

In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" sentencepiece protobuf safetensors


## 2. Imports And Runtime Check

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
from diffusers import StableDiffusion3InpaintPipeline
from PIL import Image, ImageDraw, ImageFilter, ImageChops

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Hugging Face Login

SD3.5 co the can Hugging Face token. Tren Kaggle, them secret ten `HF_TOKEN` neu model bi gated.

In [ ]:
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face with Kaggle secret HF_TOKEN.")
except Exception as exc:
    print("HF login skipped or failed. Add Kaggle secret HF_TOKEN if needed.")
    print(type(exc).__name__, exc)


## 4. Configuration

In [ ]:
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir"),
    Path("/kaggle/input/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir"),
    Path("/kaggle/input/citypersons-dataset-with-bg-image"),
]


def resolve_dataset_root(candidates):
    for root in candidates:
        if (root / "train" / "images").exists():
            return root
    return candidates[0]


DATASET_ROOT = resolve_dataset_root(DATASET_ROOT_CANDIDATES)
OUTPUT_DIR = Path("/kaggle/working/sd35_simple_augmented")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESOLUTION = 512
STRENGTH = 0.82
GUIDANCE_SCALE = 7.0
NUM_INFERENCE_STEPS = 35
SEED = 42
MASK_FEATHER_RADIUS = 2
BACKGROUND_PRESERVE_CHECK = True

print("Dataset root:", DATASET_ROOT)
print("Output dir:", OUTPUT_DIR)


## 5. Load SD3.5 Inpaint Pipeline

In [ ]:
pipe = StableDiffusion3InpaintPipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    torch_dtype=torch.float16,
    use_safetensors=True,
)

pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
print("Inpaint pipeline loaded.")


## 6. Prompts And Variants

In [ ]:
BASE_PROMPT = "CityPersons traffic camera urban street scene, realistic photo"

POSITIVE_TEMPLATE = """
{base}. Add {num_ped} {description} full-body pedestrians only inside the masked area, standing on the road or sidewalk.
Natural pose, realistic proportions, correct perspective and scale according to distance from camera,
well grounded, natural lighting, detailed clothing, sharp focus.
"""

NEGATIVE_PROMPT = """
cropped body, cut off, missing head, missing feet, floating, giant person, oversized person,
deformed, blurry, low quality, bad anatomy, bad hands, extra limbs, fused fingers,
pasted look, sticker, cartoon, painting, watermark, text, logo, changed background, altered buildings, altered cars
"""

VARIANTS = {
    "single": {"num": "one", "desc": "realistic"},
    "two": {"num": "two", "desc": "realistic"},
    "group": {"num": "three", "desc": "small group of realistic"},
}


## 7. Augmentation Function

In [ ]:
def build_pedestrian_mask(variant="single", seed=0):
    rng = random.Random(seed)
    hard_mask = Image.new("L", (RESOLUTION, RESOLUTION), 0)
    draw = ImageDraw.Draw(hard_mask)

    count = {"single": 1, "two": 2, "group": 3}[variant]
    base_xs = {
        "single": [0.50],
        "two": [0.42, 0.60],
        "group": [0.38, 0.52, 0.66],
    }[variant]

    for person_index, base_x in enumerate(base_xs[:count]):
        foot_y = rng.uniform(0.68, 0.88)
        x_center = base_x + rng.uniform(-0.04, 0.04)
        height = int(np.interp(foot_y, [0.68, 0.88], [120, 190]))
        width = int(height * 0.46)

        cx = int(x_center * RESOLUTION)
        bottom = int(foot_y * RESOLUTION)
        top = max(8, bottom - height)
        left = max(8, cx - width // 2)
        right = min(RESOLUTION - 8, cx + width // 2)
        bottom = min(RESOLUTION - 8, bottom + int(height * 0.08))

        draw.rounded_rectangle([left, top, right, bottom], radius=max(8, width // 4), fill=255)

    soft_mask = hard_mask.filter(ImageFilter.GaussianBlur(MASK_FEATHER_RADIUS))
    return hard_mask, soft_mask


def assert_background_preserved(original, composite, hard_mask):
    outside_mask = ImageChops.invert(hard_mask)
    diff = ImageChops.difference(original, composite).convert("L")
    outside_diff = Image.composite(diff, Image.new("L", diff.size, 0), outside_mask)
    extrema = outside_diff.getextrema()
    if extrema[1] != 0:
        raise RuntimeError("Background changed outside the insertion mask.")


def augment_image(image_path, variant="single", index=0):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")

    image = image.resize((RESOLUTION, RESOLUTION), Image.BICUBIC)
    hard_mask, soft_mask = build_pedestrian_mask(variant=variant, seed=SEED + index)

    prompt = POSITIVE_TEMPLATE.format(
        base=BASE_PROMPT,
        num_ped=VARIANTS[variant]["num"],
        description=VARIANTS[variant]["desc"],
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = torch.Generator(device=device).manual_seed(SEED + index)

    generated = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=image,
        mask_image=soft_mask,
        strength=STRENGTH,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        generator=generator,
    ).images[0]

    result = Image.composite(generated, image, hard_mask)
    if BACKGROUND_PRESERVE_CHECK:
        assert_background_preserved(image, result, hard_mask)

    output_path = OUTPUT_DIR / f"{image_path.stem}_aug_{variant}_{index:03d}.png"
    result.save(output_path)

    print(f"Saved: {output_path.name}")
    return output_path


## 8. Run On Dataset

In [ ]:
def run_augmentation(max_images=20, include_group=False):
    image_dir = DATASET_ROOT / "train" / "images"
    image_paths = sorted(image_dir.glob("*.jpg"))[:max_images]

    if not image_paths:
        raise FileNotFoundError(f"No .jpg images found in {image_dir}. Check DATASET_ROOT.")

    print(f"Found {len(image_paths)} images. Starting augmentation...")
    generated_paths = []

    for i, img_path in enumerate(image_paths):
        print(f"\nProcessing {i + 1}/{len(image_paths)}: {img_path.name}")

        generated_paths.append(augment_image(img_path, variant="single", index=i))

        if i % 2 == 0:
            generated_paths.append(augment_image(img_path, variant="two", index=i))

        if include_group and i % 3 == 0:
            generated_paths.append(augment_image(img_path, variant="group", index=i))

    print("\nHoan thanh!")
    print(f"Output folder: {OUTPUT_DIR}")
    return generated_paths


## 9. Start

In [ ]:
generated_paths = run_augmentation(max_images=15)
generated_paths[:5]
